# DeepTrace sprint - WORKER A
Runtime **T4 GPU**. Two copies of this notebook run in parallel (worker A and worker B) and share work through Drive.
**Runtime -> Run all.** If it disconnects, Run all again: extraction, training and evaluation all resume.
Worker A: Celeb-DF extraction + first 800 FF++ videos, then training queue A + mouth (E2) evaluation + timing.
Worker B: remaining FF++ videos, then training queue B + degradation evaluation.

In [1]:
# Cell 1 - mount Drive and install the deeptrace package from this notebook's embedded copy
from google.colab import drive
drive.mount("/content/drive")
import base64, io, tarfile, sys, os, hashlib
PKG_B64 = "<embedded deeptrace package omitted - identical to the deeptrace/ folder in this repo>"
CODE = "/content/drive/MyDrive/DeepTrace/code"
os.makedirs(CODE, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(PKG_B64)), mode="r:gz") as t:
    t.extractall(CODE)
sys.path.insert(0, CODE)
print("package version", hashlib.md5(PKG_B64.encode()).hexdigest()[:10], "->", CODE)


Mounted at /content/drive


/tmp/ipykernel_543/1045401993.py:9: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  t.extractall(CODE)


package version 41f85a5738 -> /content/drive/MyDrive/DeepTrace/code


In [2]:
# Cell 2 - setup: installs, credentials, datasets, splits
WORKER = "A"
import subprocess, sys, os, json, time, torch, pandas as pd
from pathlib import Path
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "facenet-pytorch"], check=True)
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME"); os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
import kagglehub
from deeptrace import paths
from deeptrace.data import prep
from deeptrace import train as T, experiments as X
prep.TAG = WORKER
inv = pd.read_csv(paths.MANIFESTS / "inventory.csv")
ff_rows = T.load_split("ffpp").sort_values("rel").reset_index(drop=True)          # 5,000 videos, 4 methods
cdf_rows = inv[inv.dataset == "celebdf"].sort_values("rel").reset_index(drop=True)
t0 = time.time()
CDF = next(p.parent for p in Path(kagglehub.dataset_download("reubensuju/celeb-df-v2")).rglob("List_of_testing_videos.txt"))
FF = Path(kagglehub.dataset_download("xdxd003/ff-c23"))
print(f"worker {WORKER} | data ready in {time.time()-t0:.0f}s | cpus {os.cpu_count()} | {torch.cuda.get_device_name(0)}")
print("ffpp videos:", len(ff_rows), "| celebdf videos:", len(cdf_rows))


Using Colab cache for faster access to the 'celeb-df-v2' dataset.
Using Colab cache for faster access to the 'ff-c23' dataset.
worker A | data ready in 4s | cpus 2 | Tesla T4
ffpp videos: 5000 | celebdf videos: 6529


In [3]:
# Cell 3 - extraction share of worker A (resumable)
prep.run(cdf_rows, "celebdf", CDF)
prep.run(ff_rows.iloc[:800], "ffpp", FF)


[celebdf] done 1221, to do 5308
  400/5308  1.54 vid/s  eta 53 min
  800/5308  1.50 vid/s  eta 50 min
  1200/5308  1.49 vid/s  eta 46 min
  1600/5308  1.40 vid/s  eta 44 min
  2000/5308  1.35 vid/s  eta 41 min
  2400/5308  1.34 vid/s  eta 36 min
  2800/5308  1.32 vid/s  eta 32 min
  3200/5308  1.32 vid/s  eta 27 min
  3600/5308  1.33 vid/s  eta 21 min
  4000/5308  1.31 vid/s  eta 17 min
  4400/5308  1.31 vid/s  eta 12 min
  4800/5308  1.30 vid/s  eta 7 min
  5200/5308  1.29 vid/s  eta 1 min
[celebdf] finished 5308 in 68.3 min
[ffpp] done 2424, to do 796
  400/796  0.65 vid/s  eta 10 min
[ffpp] finished 796 in 21.7 min


In [4]:
# Cell 4 - wait until BOTH workers have finished extracting (pulls the other worker's shards from Drive)
need = {"celebdf": set(cdf_rows.rel), "ffpp": set(ff_rows.rel)}
while True:
    left = {ds: len(need[ds] - prep.restore(ds)) for ds in need}
    print(time.strftime("%H:%M:%S"), "videos still being extracted:", left, flush=True)
    if all(v == 0 for v in left.values()):
        break
    time.sleep(60)
print("ALL CROPS AVAILABLE")


16:27:21 videos still being extracted: {'celebdf': 0, 'ffpp': 984}
16:28:29 videos still being extracted: {'celebdf': 0, 'ffpp': 584}
... (15 more waiting lines)
ALL CROPS AVAILABLE


In [5]:
# Cell 5 - training queue A (each run ~10-13 min on T4; finished runs are skipped)
Q = [dict(name="v1_mean",    train_on="ffpp", head="mean"),
     dict(name="v2_bilstm",  train_on="ffpp", head="bilstm"),
     dict(name="v3_freq",    train_on="ffpp", head="mean", freq=True),
     dict(name="e2_masked",  train_on="ffpp", head="mean", view="mouth_masked"),
     dict(name="v1_mean",    train_on="ffpp", head="mean", seed=1),
     dict(name="v1_mean",    train_on="ffpp", head="mean", seed=2)]
for cfg in Q:
    try:
        T.run(cfg)
    except Exception as e:
        print("RUN FAILED", cfg, repr(e))
print("QUEUE A DONE")


[v1_mean_ffpp_s0] train 3600 (2880 fake) | val 700 (560 fake)
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:431: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:437: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


  ep1/8 loss 0.2429 val_auc 0.9089 (140s)
  ep2/8 loss 0.1767 val_auc 0.9451 (123s)
  ep3/8 loss 0.1368 val_auc 0.9673 (123s)
  ep4/8 loss 0.1201 val_auc 0.9633 (129s)
  ep5/8 loss 0.0971 val_auc 0.9649 (124s)
  ep6/8 loss 0.0824 val_auc 0.9707 (127s)
  ep7/8 loss 0.0670 val_auc 0.9730 (126s)
  ep8/8 loss 0.0642 val_auc 0.9741 (128s)
  test ffpp              AUC 0.9737  bal_acc 0.910  real_rec 0.864 fake_rec 0.955
  [data] 1 videos have no face crops and are excluded: {('test', 0): 1}
  test celebdf_strict    AUC 0.8689  bal_acc 0.786  real_rec 0.739 fake_rec 0.834
  test celebdf_official  AUC 0.8215  bal_acc 0.733  real_rec 0.764 fake_rec 0.703
[v2_bilstm_ffpp_s0] train 3600 (2880 fake) | val 700 (560 fake)
  ep1/8 loss 0.2436 val_auc 0.9206 (128s)
  ep2/8 loss 0.1786 val_auc 0.9354 (127s)
  ep3/8 loss 0.1453 val_auc 0.9636 (126s)
  ep4/8 loss 0.1161 val_auc 0.9723 (129s)
  ep5/8 loss 0.0968 val_auc 0.9712 (123s)
  ep6/8 loss 0.0786 val_auc 0.9739 (122s)
  ep7/8 loss 0.0666 val_auc 0.

In [6]:
def wait_for(run_id):
    while not (paths.RUNS / run_id / "metrics.json").exists():
        print(time.strftime("%H:%M:%S"), "waiting for", run_id, flush=True); time.sleep(60)
# Cell 6 - E2 mouth evaluation (G1) and timing (G7)
X.mouth_eval("v1_mean_ffpp_s0", "ffpp")
X.mouth_eval("e2_masked_ffpp_s0", "ffpp")
wait_for("v1_mean_celebdf_strict_s0")
X.mouth_eval("v1_mean_celebdf_strict_s0", "celebdf_strict")
tim = []
for rid in ["v1_mean_ffpp_s0", "v2_bilstm_ffpp_s0", "v3_freq_ffpp_s0"]:
    tim.append(X.timing(rid))
for rid in ["v2_tdiff_ffpp_s0"]:
    wait_for(rid); tim.append(X.timing(rid))
pd.DataFrame(tim).round(2).to_csv(paths.REPORTS / "timing.csv", index=False)
print(pd.DataFrame(tim).round(2).to_string())
print("EVAL A DONE")


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:431: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:437: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


{'run': 'v1_mean_ffpp_s0', 'trained_view': 'full', 'test_view': 'full', 'test': 'ffpp', 'auc': 0.9737, 'bal_acc': 0.9098, 'auc_Deepfakes': 0.984, 'auc_Face2Face': 0.9757, 'auc_FaceSwap': 0.9887, 'auc_NeuralTextures': 0.9463, 'auc_motion_low': np.float64(0.9584), 'n_motion_low': 234, 'auc_motion_mid': np.float64(0.9673), 'n_motion_mid': 233, 'auc_motion_high': np.float64(0.975), 'n_motion_high': 233}
{'run': 'v1_mean_ffpp_s0', 'trained_view': 'full', 'test_view': 'mouth_masked', 'test': 'ffpp', 'auc': 0.7692, 'bal_acc': 0.5768, 'auc_Deepfakes': 0.8255, 'auc_Face2Face': 0.7049, 'auc_FaceSwap': 0.9242, 'auc_NeuralTextures': 0.6221, 'auc_motion_low': np.float64(0.706), 'n_motion_low': 234, 'auc_motion_mid': np.float64(0.7356), 'n_motion_mid': 233, 'auc_motion_high': np.float64(0.8067), 'n_motion_high': 233}
{'run': 'v1_mean_ffpp_s0', 'trained_view': 'full', 'test_view': 'mouth_only', 'test': 'ffpp', 'auc': 0.8174, 'bal_acc': 0.717, 'auc_Deepfakes': 0.8302, 'auc_Face2Face': 0.855, 'auc_Face

In [7]:
# Cell 7 - summary of everything finished so far (safe to run any time)
tab = T.collect()
tab = tab[~tab.name.str.startswith("smoke")]
print(tab.groupby(["name", "train_on", "test"]).agg(auc=("auc", "mean"), auc_std=("auc", "std"), n=("seed", "nunique"),
      bal_acc=("bal_acc", "mean"), real_rec=("real_rec", "mean"), fake_rec=("fake_rec", "mean")).round(4).to_string())
tab.to_csv(paths.REPORTS / "all_runs.csv", index=False)
pm = {f.parent.name: json.loads(f.read_text())["ffpp"].get("per_method_auc", {}) for f in sorted(paths.RUNS.glob("*_ffpp_s*/metrics.json"))}
print("\nFF++ test AUC per method:"); print(pd.DataFrame(pm).T.round(4).to_string())
deg = [pd.read_csv(f) for f in paths.RUNS.glob("*/degradation.csv")]
if deg:
    deg = pd.concat(deg); deg.to_csv(paths.REPORTS / "degradation_all.csv", index=False)
    print("\nDegradation AUC:"); print(deg.pivot_table(index=["run", "test"], columns="cond", values="auc").round(4).to_string())
mo = [pd.read_csv(f) for f in paths.RUNS.glob("*/mouth_eval_*.csv")]
if mo:
    mo = pd.concat(mo); mo.to_csv(paths.REPORTS / "mouth_eval_all.csv", index=False)
    print("\nMouth evaluation:"); print(mo.round(4).to_string())


                                              auc  auc_std  n  bal_acc  real_rec  fake_rec
name      train_on       test                                                             
e2_masked ffpp           celebdf_official  0.6931      NaN  1   0.6172    0.7697    0.4647
                         celebdf_strict    0.7662      NaN  1   0.6821    0.7523    0.6120
                         ffpp              0.9085      NaN  1   0.8125    0.8571    0.7679
lomo_noNT ffpp           celebdf_official  0.7951      NaN  1   0.5799    0.9775    0.1824
                         celebdf_strict    0.8547      NaN  1   0.6675    0.9771    0.3580
                         ffpp              0.9310      NaN  1   0.8545    0.9571    0.7518
robust    ffpp           celebdf_official  0.8441      NaN  1   0.7398    0.8090    0.6706
                         celebdf_strict    0.8811      NaN  1   0.7890    0.7661    0.8120
                         ffpp              0.9606      NaN  1   0.9062    0.9143    0.8982

KeyError: 'ffpp'

In [8]:
# Fixed summary: per-method AUC, degradation, mouth test, and bootstrap CIs vs Model v1
from deeptrace.evaluate import bootstrap_auc_diff
pm = {}
for f in sorted(paths.RUNS.glob("*_ffpp_s*/metrics.json")):
    m = json.loads(f.read_text())
    if "ffpp" in m and "per_method_auc" in m["ffpp"]:
        pm[f.parent.name] = m["ffpp"]["per_method_auc"]
print("FF++ test AUC per method:"); print(pd.DataFrame(pm).T.round(4).to_string())

deg = [pd.read_csv(f) for f in paths.RUNS.glob("*/degradation.csv")]
if deg:
    deg = pd.concat(deg); deg.to_csv(paths.REPORTS / "degradation_all.csv", index=False)
    print("\nDegradation AUC:"); print(deg.pivot_table(index=["run", "test"], columns="cond", values="auc").round(4).to_string())

mo = [pd.read_csv(f) for f in paths.RUNS.glob("*/mouth_eval_*.csv")]
if mo:
    mo = pd.concat(mo); mo.to_csv(paths.REPORTS / "mouth_eval_all.csv", index=False)
    print("\nMouth evaluation:"); print(mo.round(4).T.to_string())

rows = []
base = "v1_mean_ffpp_s0"
for other in ["robust_ffpp_s0", "v2_bilstm_ffpp_s0", "v2_tdiff_ffpp_s0", "v3_freq_ffpp_s0"]:
    for ts in ["ffpp", "celebdf_strict", "celebdf_official"]:
        a = pd.read_csv(paths.RUNS / base / f"preds_{ts}.csv")
        b = pd.read_csv(paths.RUNS / other / f"preds_{ts}.csv")
        j = a.merge(b, on=["dataset", "rel", "label"], suffixes=("_a", "_b"))
        d, ci = bootstrap_auc_diff(j.label, j.p_a, j.p_b)
        rows.append(dict(model=other, test=ts, auc_diff_vs_v1=round(d, 4), ci_low=round(ci[0], 4), ci_high=round(ci[1], 4),
                         significant=not (ci[0] <= 0 <= ci[1])))
ci = pd.DataFrame(rows); ci.to_csv(paths.REPORTS / "bootstrap_vs_v1.csv", index=False)
print("\nPaired bootstrap (other - v1, 95% CI):"); print(ci.to_string())
t = paths.REPORTS / "timing.csv"
if t.exists(): print("\nTiming:"); print(pd.read_csv(t).to_string())

FF++ test AUC per method:
                   Deepfakes  Face2Face  FaceSwap  NeuralTextures
e2_masked_ffpp_s0     0.9571     0.8939    0.9535          0.8296
lomo_noNT_ffpp_s0     0.9770     0.9736    0.9904          0.7829
robust_ffpp_s0        0.9726     0.9725    0.9915          0.9057
v1_mean_ffpp_s0       0.9840     0.9757    0.9887          0.9463
v1_mean_ffpp_s1       0.9805     0.9761    0.9893          0.9414
v1_mean_ffpp_s2       0.9811     0.9707    0.9831          0.9428
v2_bilstm_ffpp_s0     0.9816     0.9735    0.9896          0.9387
v2_tdiff_ffpp_s0      0.9821     0.9718    0.9855          0.9370
v3_freq_ffpp_s0       0.9746     0.9711    0.9774          0.9303

Degradation AUC:
cond                                       clean  down112  down64  jpeg30  jpeg50  jpeg50+down112
run                       test                                                                   
robust_ffpp_s0            celebdf_strict  0.8811   0.8607  0.8270  0.8417  0.8644          0.8324
  